# 02 — India Ganges Plain, Sentinel-2 with FTW Engine

Delineates smallholder agricultural fields in the Ganges Plain using Sentinel-2 imagery and the **FTW** (Fields of the World) engine with a country-specific model for India.

**Estimated runtime:** ~30–60 minutes (5 years, small AOI, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,ftw]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/india_ganges")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2020, 2025)
SOURCE = "sentinel2"
ENGINE = "ftw"

## Create Study Area

A small AOI in the Ganges Plain (Uttar Pradesh) covering smallholder agricultural fields.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [80.90, 26.80],
                        [81.00, 26.80],
                        [81.00, 26.90],
                        [80.90, 26.90],
                        [80.90, 26.80],
                    ]
                ],
            },
            "properties": {"name": "Ganges Plain AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "ganges_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation (2020–2024)

In [ ]:
all_results = {}

for year in YEARS:
    print(f"Processing {year}...", end=" ")
    output_path = OUTPUT_DIR / f"fields_s2_{year}.gpkg"

    gdf = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=year,
        engine=ENGINE,
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        composite_method="median",
        cloud_cover_max=30,
        min_area=500,
        simplify=1.0,
    )
    all_results[year] = gdf
    print(f"{len(gdf)} fields")

## Visualization

In [ ]:
if all_results:
    latest_year = max(all_results.keys())
    m = agribound.show_boundaries(
        all_results[latest_year],
        basemap="Google.Satellite",
        output_html=str(OUTPUT_DIR / "map_ganges.html"),
    )
    m